# 03 — Limpieza y construcción del panel macro Eurostat

> **Pipeline:** 01 Auditoría → 02 Limpieza ESS → **03 Eurostat** → 04 Construcción EPBI → 05 Integración micro–macro → 06 Econometría → 07 Datos para dashboard

Mientras el notebook 02 prepara la rama de **microdatos ESS**, esta etapa construye en paralelo la rama de **contexto macroeconómico**. El notebook 03 no modifica la muestra individual ni calcula el EPBI: transforma los ficheros de Eurostat en un panel homogéneo **país-año** que podrá asignarse posteriormente a cada entrevistado.

Las dos ramas —micro y macro— permanecerán separadas hasta el notebook 05.

## Criterios de construcción

1. Se procesan únicamente los indicadores Eurostat necesarios para el proyecto.
2. Cada indicador se filtra antes de integrarse para no mezclar unidades, grupos de edad, sexos o categorías estadísticas incompatibles.
3. No se promedian categorías distintas dentro de un mismo indicador.
4. La renta media se expresa exclusivamente en **euros (`EUR`)**.
5. Los códigos de país se armonizan con la codificación ESS (`EL → GR`, `UK → GB`).
6. Se eliminan los agregados supranacionales: el panel final contiene únicamente países.
7. Se conservan datos de **2007 a 2024**, de forma que el notebook 05 pueda realizar un emparejamiento exacto o, cuando sea necesario, dentro de un margen máximo de ±1 año.
8. La única salida es `eurostat_macro_panel.parquet`.


In [ ]:
from pathlib import Path
from functools import reduce
import re
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.upper() == "CODIGO" else CURRENT_DIR

BRUTOS = PROJECT_ROOT / "DATOS" / "BRUTOS"
PROCESADOS = PROJECT_ROOT / "DATOS" / "PROCESADOS"
PROCESADOS.mkdir(parents=True, exist_ok=True)

OUT_PARQUET = PROCESADOS / "eurostat_macro_panel.parquet"

# ESS analítica: 2008-2023.
# Se conserva 2007-2024 para permitir posteriormente una correspondencia macro ±1 año.
YEAR_MIN = 2007
YEAR_MAX = 2024

print("Carpeta de brutos:", BRUTOS)
print("Salida:", OUT_PARQUET)

## 1. Indicadores y filtros

La preparación comienza definiendo qué indicador representa cada fichero Eurostat y qué filtros estadísticos deben aplicarse. Esta configuración explícita garantiza que cada serie conserve una interpretación homogénea antes de combinarla con las demás.


In [ ]:
INDICATORS = {
    "renta_media_eur": {
        "file": "estat_ilc_di03.tsv.gz",
        "filters": {
            "age": ["Y_LT65"],
            "sex": ["T"],
            "statinfo": ["MEAN_EI"],
            "unit": ["EUR"],
        },
        "description": "Renta media equivalente, población menor de 65 años, euros.",
    },

    "ratio_quintiles_renta": {
        "file": "estat_ilc_di11c.tsv.gz",
        "filters": {
            "age": ["TOTAL"],
            "sex": ["T"],
            "unit": ["RAT"],
        },
        "description": "Ratio de quintiles de renta, población total.",
    },

    "gini": {
        "file": "estat_ilc_di12.tsv.gz",
        "filters": {
            "age": ["TOTAL"],
            "statinfo": ["GINI_HND"],
        },
        "description": "Índice de Gini de la renta disponible.",
    },

    "gini_antes_transferencias": {
        "file": "estat_ilc_di12b.tsv.gz",
        "filters": {
            "statinfo": ["GINI_HND"],
        },
        "description": "Índice de Gini antes de transferencias sociales.",
    },

    "gini_pensiones_transferencias": {
        "file": "estat_ilc_di12c.tsv.gz",
        "filters": {
            "statinfo": ["GINI_HND"],
        },
        "description": "Índice de Gini tras pensiones y antes de otras transferencias.",
    },

    "riesgo_pobreza": {
        "file": "estat_ilc_li02.tsv.gz",
        "filters": {
            "statinfo": ["MED_EI"],
            "unit": ["PC"],
            "rskpovth": ["B_60"],
            "sex": ["T"],
            "age": ["TOTAL"],
        },
        "description": "Tasa de riesgo de pobreza, umbral del 60 % de la renta mediana.",
    },

    "brecha_pobreza": {
        "file": "estat_ilc_li11.tsv.gz",
        "filters": {
            "statinfo": ["MED_EI"],
            "unit": ["PC"],
            "rskpovth": ["B_60"],
            "sex": ["T"],
            "age": ["TOTAL"],
        },
        "description": "Brecha relativa de riesgo de pobreza.",
    },

    "sobrecarga_vivienda": {
        "file": "estat_ilc_lvho07a.tsv.gz",
        "filters": {
            "unit": ["PC"],
            "rskpovth": ["TOTAL"],
            "age": ["TOTAL"],
            "sex": ["T"],
        },
        "description": "Tasa de sobrecarga del coste de la vivienda.",
    },

    "privacion_material_social_severa": {
        "file": "estat_ilc_mdsd11.tsv.gz",
        "filters": {
            "age": ["TOTAL"],
            "sex": ["T"],
            "unit": ["PC"],
        },
        "description": "Privación material y social severa.",
    },

    "ratio_renta_mayores": {
        "file": "estat_tespn020.tsv.gz",
        "filters": {
            "statinfo": ["R_MED_I"],
            "age": ["Y_GE65"],
            "sex": ["T"],
        },
        "description": "Ratio de renta mediana de la población de 65 años o más.",
    },
}

config = pd.DataFrame([
    {
        "variable": variable,
        "fichero": cfg["file"],
        "filtros": str(cfg["filters"]),
        "descripcion": cfg["description"],
    }
    for variable, cfg in INDICATORS.items()
])

display(config)

## 2. Comprobación de ficheros necesarios

Antes de procesar ninguna serie se verifica que estén disponibles todos los ficheros requeridos. Si falta una fuente, la ejecución se detiene para evitar construir un panel macro incompleto de forma inadvertida.


In [ ]:
required_files = [BRUTOS / cfg["file"] for cfg in INDICATORS.values()]
missing_files = [p.name for p in required_files if not p.exists()]

if missing_files:
    raise FileNotFoundError(
        "Faltan ficheros Eurostat necesarios:\n- " + "\n- ".join(missing_files)
    )

print("Ficheros necesarios encontrados:", len(required_files))

## 3. Funciones de lectura y filtrado

Una vez comprobadas las fuentes, se centraliza la lógica de lectura, filtrado, conversión temporal y armonización de códigos. Utilizar las mismas funciones para todos los indicadores reduce diferencias de tratamiento entre series.


In [ ]:
def clean_name(value):
    value = str(value).strip()
    value = re.sub(r"[^A-Za-z0-9_]+", "_", value)
    return value.strip("_").lower()


def parse_value(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    if text in {"", ":"}:
        return np.nan

    text = text.replace(",", ".")

    # Eurostat puede añadir flags después del valor (p, e, b, etc.).
    match = re.search(r"[-+]?\d+(?:\.\d+)?", text)
    return float(match.group()) if match else np.nan


def read_eurostat_tsv(path):
    raw = pd.read_csv(
        path,
        sep="\t",
        compression="gzip",
        dtype=str,
        engine="c",
    )

    raw.columns = [str(c).strip() for c in raw.columns]
    first_column = raw.columns[0]

    dimension_header = re.sub(
        r"\\TIME_PERIOD$",
        "",
        first_column,
        flags=re.IGNORECASE,
    )

    dimensions = [clean_name(x) for x in dimension_header.split(",")]

    dimension_data = (
        raw[first_column]
        .astype("string")
        .str.strip()
        .str.split(",", expand=True)
    )

    if dimension_data.shape[1] != len(dimensions):
        raise ValueError(
            f"{path.name}: dimensiones incompatibles entre cabecera y datos."
        )

    dimension_data.columns = dimensions

    year_mapping = {
        c: str(c).strip()
        for c in raw.columns[1:]
        if re.fullmatch(r"\d{4}", str(c).strip())
    }

    if not year_mapping:
        raise ValueError(f"{path.name}: no se detectaron columnas anuales.")

    values = raw[list(year_mapping)].rename(columns=year_mapping)

    wide = pd.concat(
        [dimension_data.reset_index(drop=True), values.reset_index(drop=True)],
        axis=1,
    )

    years = list(year_mapping.values())

    long = wide.melt(
        id_vars=dimensions,
        value_vars=years,
        var_name="year",
        value_name="raw_value",
    )

    long["year"] = pd.to_numeric(long["year"], errors="coerce").astype("Int64")
    long["value"] = long["raw_value"].map(parse_value)

    return long, dimensions


def apply_filters(data, filters, indicator):
    out = data.copy()

    for dimension, accepted in filters.items():
        if dimension not in out.columns:
            raise KeyError(
                f"{indicator}: el fichero no contiene la dimensión {dimension!r}."
            )

        accepted = [str(x) for x in accepted]
        available = set(out[dimension].dropna().astype(str).unique())

        missing_categories = set(accepted) - available
        if missing_categories:
            raise ValueError(
                f"{indicator}: categorías solicitadas no disponibles en {dimension}: "
                f"{sorted(missing_categories)}"
            )

        out = out.loc[
            out[dimension].astype(str).isin(accepted)
        ].copy()

    if out.empty:
        raise ValueError(f"{indicator}: los filtros no dejan ninguna observación.")

    return out


def build_indicator(variable, cfg):
    path = BRUTOS / cfg["file"]
    data, dimensions = read_eurostat_tsv(path)

    filtered = apply_filters(data, cfg["filters"], variable)

    # Tras filtrar, no puede quedar más de una categoría en ninguna dimensión
    # distinta de país. De esta forma nunca se promedian categorías incompatibles.
    remaining_multiple = {
        dim: sorted(filtered[dim].dropna().astype(str).unique().tolist())
        for dim in dimensions
        if dim != "geo" and filtered[dim].nunique(dropna=True) > 1
    }

    if remaining_multiple:
        raise ValueError(
            f"{variable}: quedan dimensiones con múltiples categorías: "
            f"{remaining_multiple}"
        )

    filtered = filtered.loc[
        filtered["year"].between(YEAR_MIN, YEAR_MAX)
    ].copy()

    filtered["geo"] = (
        filtered["geo"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    result = (
        filtered
        .dropna(subset=["geo", "year", "value"])
        .groupby(["geo", "year"], as_index=False)["value"]
        .mean()
        .rename(columns={"geo": "cntry", "value": variable})
    )

    if result.duplicated(["cntry", "year"]).any():
        raise ValueError(f"{variable}: se han generado duplicados país-año.")

    return result, dimensions, len(data), len(filtered)

## 4. Procesamiento de indicadores

Cada indicador se procesa de forma independiente con sus filtros específicos. El resultado intermedio de esta etapa es una colección de series limpias y comparables, cada una con una única observación por país y año.


In [ ]:
tables = []
processing_log = []

for variable, cfg in INDICATORS.items():
    try:
        table, dimensions, n_raw, n_filtered = build_indicator(variable, cfg)
        tables.append(table)

        processing_log.append({
            "variable": variable,
            "estado": "OK",
            "dimensiones": ", ".join(dimensions),
            "filas_largas_originales": n_raw,
            "filas_tras_filtros": n_filtered,
            "pais_ano_validos": len(table),
            "paises": table["cntry"].nunique(),
            "primer_ano": table["year"].min(),
            "ultimo_ano": table["year"].max(),
        })

    except Exception as exc:
        processing_log.append({
            "variable": variable,
            "estado": "ERROR",
            "detalle": repr(exc),
        })

processing_log = pd.DataFrame(processing_log)
display(processing_log)

errors = processing_log.loc[processing_log["estado"].eq("ERROR")]

if not errors.empty:
    raise RuntimeError(
        "Hay indicadores Eurostat que no se han procesado correctamente. "
        "Revisar la tabla anterior antes de continuar."
    )

assert len(tables) == len(INDICATORS)

## 5. Construcción del panel país-año

Con las series ya depuradas, se integran en una única estructura macroeconómica. La clave del panel es `cntry + year`, y cada fila representa el contexto de un país en un año determinado.


In [ ]:
eurostat = reduce(
    lambda left, right: pd.merge(
        left,
        right,
        on=["cntry", "year"],
        how="outer",
        validate="one_to_one",
    ),
    tables,
)

# Armonización de códigos Eurostat → ESS
COUNTRY_CODE_MAP = {
    "EL": "GR",
    "UK": "GB",
}

eurostat["cntry"] = eurostat["cntry"].replace(COUNTRY_CODE_MAP)

# Si el cambio de código generase duplicados, se detiene la ejecución.
if eurostat.duplicated(["cntry", "year"]).any():
    duplicates = eurostat.loc[
        eurostat.duplicated(["cntry", "year"], keep=False)
    ].sort_values(["cntry", "year"])
    display(duplicates)
    raise ValueError("La armonización de códigos ha producido duplicados país-año.")

# Eurostat contiene agregados como EU27_2020, EA20, etc.
# El análisis ESS necesita únicamente países.
# Los códigos ESS/ISO de países tienen dos letras.
country_mask = eurostat["cntry"].astype("string").str.fullmatch(r"[A-Z]{2}", na=False)
excluded_aggregates = eurostat.loc[~country_mask, "cntry"].dropna().unique().tolist()

eurostat = eurostat.loc[country_mask].copy()

eurostat["year"] = pd.to_numeric(eurostat["year"], errors="coerce").astype("Int64")

eurostat = (
    eurostat
    .sort_values(["cntry", "year"])
    .reset_index(drop=True)
)

print("Dimensiones del panel:", eurostat.shape)
print("Países:", eurostat["cntry"].nunique())
print("Periodo:", eurostat["year"].min(), "-", eurostat["year"].max())
print("Agregados Eurostat eliminados:", excluded_aggregates)

display(eurostat.head(15))

## 6. Cobertura y controles finales

Antes de exportar se comprueba la unicidad de la clave país-año, la armonización de códigos, el intervalo temporal y la cobertura de cada indicador. Estos controles son importantes porque el notebook 05 asumirá que este panel no contiene duplicados por contexto.


In [ ]:
macro_vars = list(INDICATORS.keys())

coverage = pd.DataFrame([
    {
        "variable": var,
        "validos": int(eurostat[var].notna().sum()),
        "pct_validos": round(100 * eurostat[var].notna().mean(), 2),
        "paises_con_datos": int(eurostat.loc[eurostat[var].notna(), "cntry"].nunique()),
        "primer_ano": eurostat.loc[eurostat[var].notna(), "year"].min(),
        "ultimo_ano": eurostat.loc[eurostat[var].notna(), "year"].max(),
    }
    for var in macro_vars
])

display(coverage)

checks = pd.DataFrame([
    {
        "control": "Clave cntry-year única",
        "resultado": "OK" if eurostat.duplicated(["cntry", "year"]).sum() == 0 else "REVISAR",
    },
    {
        "control": "Solo códigos país de dos letras",
        "resultado": "OK" if eurostat["cntry"].astype("string").str.fullmatch(r"[A-Z]{2}", na=False).all() else "REVISAR",
    },
    {
        "control": "Código Grecia armonizado a GR",
        "resultado": "OK" if "EL" not in set(eurostat["cntry"].dropna()) else "REVISAR",
    },
    {
        "control": "Código Reino Unido armonizado a GB",
        "resultado": "OK" if "UK" not in set(eurostat["cntry"].dropna()) else "REVISAR",
    },
    {
        "control": "Años dentro del rango esperado",
        "resultado": "OK" if eurostat["year"].between(YEAR_MIN, YEAR_MAX).all() else "REVISAR",
    },
])

display(checks)

if (checks["resultado"] != "OK").any():
    raise ValueError("Alguno de los controles finales requiere revisión.")

## 7. Exportación

Superados los controles, se guarda únicamente el panel necesario para las etapas posteriores. No se generan copias Excel, CSV ni ficheros auxiliares, de modo que exista una única fuente macroeconómica de referencia.


In [ ]:
eurostat.to_parquet(OUT_PARQUET, index=False)

print("Panel Eurostat guardado:")
print(" -", OUT_PARQUET)
print("Filas:", len(eurostat))
print("Columnas:", eurostat.shape[1])

## 8. Resultado de la fase 03 y continuidad

La salida de esta rama es:

`DATOS/PROCESADOS/eurostat_macro_panel.parquet`

Su unidad es **país-año** y contiene:

- `renta_media_eur`
- `ratio_quintiles_renta`
- `gini`
- `gini_antes_transferencias`
- `gini_pensiones_transferencias`
- `riesgo_pobreza`
- `brecha_pobreza`
- `sobrecarga_vivienda`
- `privacion_material_social_severa`
- `ratio_renta_mayores`

Este fichero queda preparado a la espera de la integración. El **notebook 04** vuelve a la rama individual para construir el EPBI a partir de la salida del notebook 02; después, el **notebook 05** combinará ambos resultados.
